In [2]:
library(Seurat)
library(stringr)
library(parallel)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t




In [12]:
# Carpeta con todos los archivos
base_dir <- "/home/nmedina/Nico/David/proteinas_T4/udec_alzheimer-main/sncell/data/GSE268599_RAW"

# Listar todos los archivos
all_files <- list.files(base_dir, full.names = TRUE)

# Extraer IDs únicos
ids <- unique(str_extract(basename(all_files), "^GSM[0-9]+"))

# Función para procesar cada ID
filtrar_GSM <- function(id, base_dir, min_features = 200, max_features = 2500, max_mt = 5) {
  
  # Buscar los archivos de este ID
  pattern <- paste0("^", id)
  files_id <- list.files(base_dir, full.names = TRUE, pattern = pattern)
  
  # Crear carpeta temporal
  tmp_dir <- file.path(tempdir(), id)
  dir.create(tmp_dir, showWarnings = FALSE)
  
  # Copiar y renombrar a los nombres que espera Read10X
  file.copy(files_id[grepl("barcodes", files_id)], file.path(tmp_dir, "barcodes.tsv.gz"), overwrite = TRUE)
  file.copy(files_id[grepl("features", files_id)], file.path(tmp_dir, "features.tsv.gz"), overwrite = TRUE)
  file.copy(files_id[grepl("matrix",   files_id)], file.path(tmp_dir, "matrix.mtx.gz"),   overwrite = TRUE)
  
  # Leer datos
  sc.data <- Read10X(data.dir = tmp_dir)
  
  # Crear objeto Seurat
  seu <- CreateSeuratObject(
    counts = sc.data,
    project = id,
    min.cells = 3,
    min.features = min_features
  )
  
  # Calcular % mitocondrial
  seu[["percent.mt"]] <- PercentageFeatureSet(seu, pattern = "^MT-")
  
  # Filtrar células
  seu <- subset(seu,
                subset = nFeature_RNA > min_features &
                         nFeature_RNA < max_features &
                         percent.mt < max_mt)
  
  # Guardar resultado en disco
  saveRDS(seu, file = file.path("/home/nmedina/Nico/David/proteinas_T4/udec_alzheimer-main/sncell/results/datos_filtrados/", paste0(id, "_seurat_filtered.rds")))
  
  return(id)
}


In [13]:
# paralelizamos para ahorrar tiempo 
# Detectar núcleos disponibles
numCores <- detectCores() - 1  # deja 1 libre para el sistema


# ---- Procesar todos los IDs ----
resultados <- mclapply(ids,
                       filtrar_GSM,
                       base_dir = base_dir,
                       mc.cores = numCores)

print("Procesamiento paralelo completado")

[1] "Procesamiento paralelo completado"
